In [5]:
import os

print(os.listdir('/kaggle/input'))

['datasets']


In [6]:
import torch

print(torch.cuda.is_available())

True


In [7]:
import os

print(os.listdir("/kaggle/input"))

['datasets']


In [8]:
print(os.listdir("/kaggle/input/datasets"))

['sudhirkh']


In [9]:
print(os.listdir("/kaggle/input/datasets/sudhirkh"))

['deepfashion-preprocessed', 'deepfashion-preprocessed-val', 'deepfashion2']


In [10]:
DATA_ROOT = "/kaggle/input/datasets/sudhirkh/deepfashion2/DeepFashion2"

train_image_folder = DATA_ROOT + "/train/image"
train_json_folder  = DATA_ROOT + "/train/annos"

val_image_folder = DATA_ROOT + "/validation/image"
val_json_folder  = DATA_ROOT + "/validation/annos"

test_image_folder = DATA_ROOT + "/test/image"

In [11]:
import pickle

with open("/kaggle/input/datasets/sudhirkh/deepfashion-preprocessed/train_data.pkl", "rb") as f:
    image_paths, labels = pickle.load(f)

print("Loaded:", len(image_paths))

Loaded: 144174


In [12]:
import pickle

with open("/kaggle/input/datasets/sudhirkh/deepfashion-preprocessed-val/validation_data.pkl", "rb") as f:
    val_image_paths, val_labels = pickle.load(f)

print("Loaded:", len(val_image_paths))

Loaded: 23741


In [13]:
from sklearn.model_selection import train_test_split

train_paths, _, train_labels, _ = train_test_split(
    image_paths,
    labels,
    test_size = 0.5,
    random_state=42
)

print("Train size:",len(train_paths))

Train size: 72087


In [14]:
train_paths = image_paths
train_labels = labels

print("Train size:", len(train_paths))

Train size: 144174


In [15]:
from torchvision import transforms
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [16]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class FashionDataset(Dataset):

    def __init__(self, image_paths, labels, transform=None):

        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(self.image_paths[idx]).convert("RGB")

        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

In [17]:
train_dataset = FashionDataset(
    train_paths,
    train_labels,
    transform=train_transform
)

In [18]:
val_dataset = FashionDataset(
    val_image_paths,
    val_labels,
    transform=val_transform
)

In [19]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [20]:
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [21]:
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT

model = resnet50(weights=weights)

model

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 215MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [22]:
import torch.nn as nn

in_features = model.fc.in_features
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 5)
)

model = model.to(device)

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [23]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [24]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score

epochs = 5
best_val_f1 = 0

SAVE_PATH = "/kaggle/working/best_model_resnet_full_dataset.pth"

for epoch in range(epochs):

    # ======================
    # TRAIN
    # ======================
    model.train()
    train_loss = 0
    train_preds = []
    train_labels_all = []

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # convert logits → probabilities
        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int()

        train_preds.append(preds.cpu().numpy())
        train_labels_all.append(labels.cpu().numpy())

    train_loss /= len(train_loader)

    train_preds = np.vstack(train_preds)
    train_labels_all = np.vstack(train_labels_all)

    train_f1 = f1_score(train_labels_all, train_preds, average="macro")


    # ======================
    # VALIDATION
    # ======================
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels_all = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            probs = torch.sigmoid(outputs)

            preds = (probs > 0.5).int()

            val_preds.append(preds.cpu().numpy())
            val_labels_all.append(labels.cpu().numpy())

    val_loss /= len(val_loader)

    val_preds = np.vstack(val_preds)
    val_labels_all = np.vstack(val_labels_all)

    val_f1 = f1_score(val_labels_all, val_preds, average="macro")


    # ======================
    # PRINT RESULTS
    # ======================
    print("\n---------------------------")
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train F1   : {train_f1:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val F1     : {val_f1:.4f}")


    # ======================
    # SAVE BEST MODEL
    # ======================
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        torch.save(model.state_dict(), SAVE_PATH)

        print("✅ Best model saved!")

100%|██████████| 4506/4506 [26:28<00:00,  2.84it/s]



---------------------------
Epoch 1/5
Train Loss : 0.2417
Train F1   : 0.8204
Val Loss   : 0.1983
Val F1     : 0.8590
✅ Best model saved!


100%|██████████| 4506/4506 [26:35<00:00,  2.82it/s]



---------------------------
Epoch 2/5
Train Loss : 0.1636
Train F1   : 0.8889
Val Loss   : 0.2077
Val F1     : 0.8541


100%|██████████| 4506/4506 [26:34<00:00,  2.83it/s]



---------------------------
Epoch 3/5
Train Loss : 0.1271
Train F1   : 0.9168
Val Loss   : 0.2133
Val F1     : 0.8575


100%|██████████| 4506/4506 [26:33<00:00,  2.83it/s]



---------------------------
Epoch 4/5
Train Loss : 0.1024
Train F1   : 0.9344
Val Loss   : 0.2160
Val F1     : 0.8614
✅ Best model saved!


100%|██████████| 4506/4506 [26:34<00:00,  2.83it/s]



---------------------------
Epoch 5/5
Train Loss : 0.0844
Train F1   : 0.9471
Val Loss   : 0.2341
Val F1     : 0.8644
✅ Best model saved!


In [25]:
model = resnet50(weights=None)

In [26]:
import torch
import torch.nn as nn

in_features = model.fc.in_features
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 5)
)

model = model.to(device)

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [27]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [28]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score

epochs = 15
best_val_f1 = 0

SAVE_PATH = "/kaggle/working/best_model_resnet_from_scratch.pth"

for epoch in range(epochs):

    # ======================
    # TRAIN
    # ======================
    model.train()
    train_loss = 0
    train_preds = []
    train_labels_all = []

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # convert logits → probabilities
        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int()

        train_preds.append(preds.cpu().numpy())
        train_labels_all.append(labels.cpu().numpy())

    train_loss /= len(train_loader)

    train_preds = np.vstack(train_preds)
    train_labels_all = np.vstack(train_labels_all)

    train_f1 = f1_score(train_labels_all, train_preds, average="macro")


    # ======================
    # VALIDATION
    # ======================
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels_all = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            probs = torch.sigmoid(outputs)

            preds = (probs > 0.5).int()

            val_preds.append(preds.cpu().numpy())
            val_labels_all.append(labels.cpu().numpy())

    val_loss /= len(val_loader)

    val_preds = np.vstack(val_preds)
    val_labels_all = np.vstack(val_labels_all)

    val_f1 = f1_score(val_labels_all, val_preds, average="macro")


    # ======================
    # PRINT RESULTS
    # ======================
    print("\n---------------------------")
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train F1   : {train_f1:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val F1     : {val_f1:.4f}")


    # ======================
    # SAVE BEST MODEL
    # ======================
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        torch.save(model.state_dict(), SAVE_PATH)

        print("✅ Best model saved!")

100%|██████████| 4506/4506 [26:55<00:00,  2.79it/s]



---------------------------
Epoch 1/15
Train Loss : 0.5383
Train F1   : 0.3243
Val Loss   : 0.5165
Val F1     : 0.3664
✅ Best model saved!


100%|██████████| 4506/4506 [26:54<00:00,  2.79it/s]



---------------------------
Epoch 2/15
Train Loss : 0.4517
Train F1   : 0.5317
Val Loss   : 0.4362
Val F1     : 0.5558
✅ Best model saved!


100%|██████████| 4506/4506 [27:00<00:00,  2.78it/s]


...


---------------------------
Epoch 13/15
Train Loss : 0.1957
Train F1   : 0.8656
Val Loss   : 0.3299
Val F1     : 0.7686


100%|██████████| 4506/4506 [27:03<00:00,  2.78it/s]



---------------------------
Epoch 14/15
Train Loss : 0.1824
Train F1   : 0.8764
Val Loss   : 0.3493
Val F1     : 0.7699


100%|██████████| 4506/4506 [26:54<00:00,  2.79it/s]



---------------------------
Epoch 15/15
Train Loss : 0.1718
Train F1   : 0.8852
Val Loss   : 0.3291
Val F1     : 0.7811
✅ Best model saved!
